In [ ]:
import numpy as np
import pandas as pd 
path = r"C:\Users\razib\Downloads\listings.csv\listings.csv"
listings = pd.read_csv(path, low_memory = False)
listings.head()
listings.info()
listings.columns.tolist()
listings.head(5)
listings_small = listings[['id','host_id',
                           'neighbourhood_cleansed','latitude', 'longitude',
                           'room_type','accommodates','bedrooms',
                           'price','availability_365','availability_90',
                           'review_scores_rating','review_scores_location','review_scores_value','number_of_reviews',
                           'host_is_superhost','calculated_host_listings_count']].copy()
listings_small.columns.tolist()
listings_small.info()
listings_small.to_csv(r"C:\Users\razib\Downloads\listings_clean_v1.csv", index=False)
listings_small = pd.read_csv(r"C:\Users\razib\Downloads\listings_clean_v1.csv")
listings_small.shape
#cleaning price column 
listings_small['price'] = listings_small['price'].str.replace('$','',regex=False).str.replace(',','').astype(float)
listings_small['price'].describe() #max 75000.00!!
#print(listings_small['price'].isna().sum()) #32049 NaN!!
listings_small.loc[listings_small['price'] == 75000.00]
listings_small['price'].describe()
listings_small['price_missing'] =listings_small['price'].isna() 
listings_small.groupby('price_missing').agg(availability_mean =('availability_365','mean'),reviews_mean=('number_of_reviews', 'mean'))
listings_small['price'].sort_values(ascending=False).head(50)
listings_small['price'].quantile(0.99)
(listings_small['price'] > 1000).sum()
listings_small['price_outlier'] = listings_small['price'] > listings_small['price'].quantile(0.99)
#print(listings_small.groupby('price_outlier')['price'].median())
#print(listings_small['price'].median())
listings_small['host_is_superhost'].value_counts(dropna=False)
listings_small['host_is_superhost']=listings_small['host_is_superhost'].map({'t': True,'f':False})
listings_small['host_is_superhost'].value_counts(dropna=False)
listings_small['id'].duplicated().sum()
listings_small.isna().sum()
missing = listings_small['review_scores_rating'].isna()
#print(missing.sum())
no_review = listings_small['number_of_reviews']==0
#print((no_review & missing).sum())
listings_small.to_csv(r'C:\Users\razib\Downloads\listings_clean_v2.csv', index=False)

#calender
cal_path = r"C:\Users\razib\Downloads\calendar.csv\calendar.csv"
cal = pd.read_csv(cal_path, usecols=['listing_id','date','available'],
                 parse_dates=['date'],
                 true_values = ['t'],
                 false_values = ['f'])
cal.shape
cal['available'].value_counts(dropna=False)
daily = cal.groupby('date')['available'].mean().reset_index()
daily['month'] = daily['date'].dt.strftime('%Y-%m')
monthly = daily.groupby('month')['available'].mean().reset_index()
monthly = monthly.sort_values('available')
print(monthly)
monthly.to_csv(r'C:\Users\razib\Downloads\availability_by_month.csv', index=False)
daily.to_csv(r'C:\Users\razib\Downloads\availability_daily.csv', index=False)
per_listing = cal.groupby('listing_id')['available'].mean().reset_index()
per_listing.to_csv(r'C:\Users\razib\Downloads\per_listing.csv', index=False)
rev_path = r"C:\Users\razib\Downloads\reviews.csv\reviews.csv"
reviews = pd.read_csv(rev_path, usecols=['id','listing_id','date'],parse_dates = ['date'])
reviews.shape
reviews.isna().sum()
print(reviews.duplicated().sum())
#print(reviews['id'].duplicated().sum())
reviews['month'] = reviews['date'].dt.strftime('%Y-%m')
reviews_by_month = reviews.groupby('month').size().reset_index(name='review_count')
print(reviews_by_month)
reviews.to_csv(r'C:\Users\razib\Downloads\reviews_clean.csv', index=False)
reviews_by_month.to_csv(r'C:\Users\razib\Downloads\reviews_by_month.csv', index=False)